# Create the main life-cycle inventory database in your brightway project

In [ ]:
import bw2data as bd
import bw2io as bi
from premise import *
from mescal import *

In [ ]:
ei_version = '3.12' # version of the ecoinvent database.
ei_db_name = f"ecoinvent-{ei_version}-cutoff" # name of the ecoinvent database in the brightway project
bd.projects.set_current('pypsa') # set the current brightway project
biosphere_name= 'biosphere3' # name of the biosphere database in the brightway project

### Premise parameters ###
model = 'remind-eu' # IAM used to create the database
pathway = 'SSP2-NDC' # SSP-RCP scenario used to create the database
years = [2020, 2030, 2040, 2050] # years of the database

# Or, define a complete list of scenarios
use_scenarios_list = False
scenarios = [
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [ ]:
if ei_version == '3.10.1':
    ei_version_premise = '3.10'
else:
    ei_version_premise = ei_version

## Import ecoinvent

In [ ]:
if f'ecoinvent-{ei_version}-cutoff' in bd.databases:
    print(f'ecoinvent {ei_version} is already present in the project')
else:
    bi.import_ecoinvent_release(
        version=ei_version,
        system_model='cutoff', # can be cutoff / apos / consequential / EN15804
        username='JohnDoe',
        password='1234',
        biosphere_name=biosphere_name,
    )

## Import premise databases

In [ ]:
# Clear cache is encouraged if updating premise or if encountering issues with inventories
clear_cache()

In [ ]:
# Initialize the premise database
ndb = NewDatabase(
    scenarios=scenarios if use_scenarios_list else [{"model": model, "pathway": pathway, "year": year} for year in years],
    source_db=ei_db_name,
    source_version=ei_version_premise,
    key='xxx', # ask a key to Romain Sacchi (romain.sacchi@psi.ch)
    biosphere_name=biosphere_name,
)

In [ ]:
# Update the database with IAM data
ndb.update()

In [ ]:
databases_names = (
    [f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}' for model, pathway, year in [(s['model'], s['pathway'], s['year']) for s in scenarios]] if use_scenarios_list
    else [f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}' for year in years]
)

In [ ]:
# Write the database in your brightway project
ndb.write_db_to_brightway(name=databases_names)

## Regionalization of the database

To regionalize the database, clone the [Regiopremise](https://github.com/matthieu-str/Regiopremise) repository and run the [demo.ipynb](https://github.com/matthieu-str/Regiopremise/blob/ei3.10/doc/demo.ipynb) notebook. 

## Archiving, Restoring, and Sharing Projects

In [ ]:
bi.backup.backup_project_directory(
    project=f'ecoinvent{ei_version}',
)

In [ ]:
bi.backup.restore_project_directory(
    fp='./export/backup/file_name.tar.gz',
    project_name=f'ecoinvent{ei_version}',
    overwrite_existing=True,
)

## Import additional LCIA methods

You can import LCIA methoda that are not in your default brightway project. For instance, the IMPACT World+ (expert or footprint), can be downloaded from [Zenodo](https://zenodo.org/records/18892673).

Here, we import the -1/+1 version of IMPACT World+ (Midpoint and Damage) for ecoinvent 3.12.

In [ ]:
bi.BW2Package.import_file("impact_world_plus_221-incl-CO2-uptake_brightway2_expert_version_ei312.d41d4cf583baa6b675e7bff60d276ff8.bw2package")

If your biosphere database is not named `biosphere3`, you need to run the following:

In [ ]:
biosphere_db_name = f'ecoinvent-{ei_version}-biosphere'  # the name of your biosphere database

iw_methods = [i for i in bd.methods if i[0] in [  # the name of the LCIA methods to adapt
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
]]

for method in iw_methods:
    method = bd.Method(method)
    cf_list = method.load()
    new_cf_list = []
    for cf in cf_list:
        if cf[0][0] == 'biosphere3':
            new_cf = ((biosphere_db_name, cf[0][1]),
                      cf[1])  # replace 'biosphere3' with the name of your biosphere database
            new_cf_list.append(new_cf)
        else:
            new_cf_list.append(cf)
    method.write(new_cf_list)  # overwrite the existing method

Moreover, we add a "total" category for climate change that sums the different sub-categories existing in IMPACT World+ (for conveniance)

In [ ]:
iw_methods_co2_uptake_damage = [
    i for i in bd.methods if
    i[0] == 'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)'
    and 'Climate change' in i[-1]
]

meq_ccst_cf_list = []
teq_ccst_cf_list = []
hh_ccst_cf_list = []
meq_cclt_cf_list = []
teq_cclt_cf_list = []
hh_cclt_cf_list = []

for method in iw_methods_co2_uptake_damage:
    method = bd.Method(method)
    if 'Climate change, ecosystem quality, marine ecosystem, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            meq_ccst_cf_list.append(cf)
    elif 'Climate change, ecosystem quality, marine ecosystem, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            meq_cclt_cf_list.append(cf)
    elif 'Climate change, ecosystem quality, terrestrial ecosystem, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            teq_ccst_cf_list.append(cf)
    elif 'Climate change, ecosystem quality, terrestrial ecosystem, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            teq_cclt_cf_list.append(cf)
    elif 'Climate change, human health, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            hh_ccst_cf_list.append(cf)
    elif 'Climate change, human health, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            hh_cclt_cf_list.append(cf)

In [ ]:
iw_methods_co2_uptake_midpoint = [i for i in bd.methods if i[0] == 'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)' and 'Climate change' in i[-1]]
ccst_cf_list = []
cclt_cf_list = []
for method in iw_methods_co2_uptake_midpoint:
    method = bd.Method(method)
    if 'Climate change, short term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            ccst_cf_list.append(cf)
    elif 'Climate change, long term' in method.name[-1]:
        cf_list = method.load()
        for cf in cf_list:
            cclt_cf_list.append(cf)

In [ ]:
meq_ccst_method = bd.Method((
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Ecosystem quality',
    'Climate change, ecosystem quality, marine ecosystem, short term (beta), total',
))
meq_cclt_method = bd.Method((
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Ecosystem quality',
    'Climate change, ecosystem quality, marine ecosystem, long term (beta), total',
))
teq_ccst_method = bd.Method((
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Ecosystem quality',
    'Climate change, ecosystem quality, terrestrial ecosystem, short term, total',
))
teq_cclt_method = bd.Method((
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Ecosystem quality',
    'Climate change, ecosystem quality, terrestrial ecosystem, long term, total',
))
hh_ccst_method = bd.Method((
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Human health',
    'Climate change, human health, short term, total',
))
hh_cclt_method = bd.Method((
    'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Human health',
    'Climate change, human health, long term, total',
))
ccst_method = bd.Method((
    'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Midpoint',
    'Climate change, short term, total',
))
cclt_method = bd.Method((
    'IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)',
    'Midpoint',
    'Climate change, long term, total',
))

In [ ]:
meq_ccst_method.register(**{'unit': 'PDF.m2.yr'})
meq_cclt_method.register(**{'unit': 'PDF.m2.yr'})
teq_ccst_method.register(**{'unit': 'PDF.m2.yr'})
teq_cclt_method.register(**{'unit': 'PDF.m2.yr'})
hh_ccst_method.register(**{'unit': 'DALY'})
hh_cclt_method.register(**{'unit': 'DALY'})
ccst_method.register(**{'unit': 'kg CO2-eq (short)'})
cclt_method.register(**{'unit': 'kg CO2-eq (long)'})

In [ ]:
meq_ccst_method.write(meq_ccst_cf_list)
meq_cclt_method.write(meq_cclt_cf_list)
teq_ccst_method.write(teq_ccst_cf_list)
teq_cclt_method.write(teq_cclt_cf_list)
hh_ccst_method.write(hh_ccst_cf_list)
hh_cclt_method.write(hh_cclt_cf_list)
ccst_method.write(ccst_cf_list)
cclt_method.write(cclt_cf_list)